# 📊 Extraction Data Points depuis 10-K Filings

Ce notebook extrait les informations structurées depuis les rapports 10-K HTML.

**Objectif** : Extraire pour chaque entreprise :
- Name, type
- Geography (pays, régions)
- Company NA/EU (présence)
- Segments (segments d'affaires)
- Supply chain (fournisseurs, dépendances)

**Format de sortie** : JSON structuré (un fichier par ticker ou un gros JSON)


## Étape 1 : Installation et Imports


In [1]:
# Installer les dépendances si nécessaire
# %pip install boto3 instructor[bedrock] pydantic pandas beautifulsoup4

import boto3
import instructor
from pydantic import BaseModel, Field
from typing import List, Optional
import json
import os
import re  # Pour les regex dans extraction quantitative
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup


## Étape 2 : Définir le Modèle de Données (Pydantic)


In [2]:
# Modèle de données pour extraire les informations d'un 10-K
class CompanyDataPoints(BaseModel):
    """Structure de données extraites depuis un 10-K"""
    ticker: str = Field(description="Ticker symbol de l'entreprise")
    company_name: str = Field(description="Nom complet de l'entreprise")
    company_type: Optional[str] = Field(None, description="Type d'entreprise (secteur principal)")
    
    geography: dict = Field(description="Informations géographiques", default_factory=dict)
    # Structure: {
    #   "regions": ["North America", "Europe", "Asia Pacific"],
    #   "countries": ["USA", "China", "Germany", ...],
    #   "has_na": True/False,
    #   "has_eu": True/False,
    #   "revenue_by_region": {"North America": 0.42, "Europe": 0.28, "Asia Pacific": 0.30},  # Pourcentages si disponible
    #   "assets_by_region": {"USA": 0.50, "China": 0.35, "Other": 0.15}  # Pourcentages si disponible
    # }
    
    segments: List[str] = Field(default_factory=list, description="Segments d'affaires (ex: iPhone, Mac, Services)")
    
    supply_chain: dict = Field(description="Informations supply chain", default_factory=dict)
    # Structure: {
    #   "key_suppliers": ["Foxconn", "TSMC", ...],
    #   "manufacturing_locations": ["China", "USA", ...],
    #   "dependencies": ["Component sourcing in China", ...],
    #   "risks": ["Supply chain concentration", ...],
    #   "supplier_concentration": {"Foxconn": 0.35, "TSMC": 0.25, "Other": 0.40}  # Pourcentages si disponible
    # }
    
    regulatory_risks: List[str] = Field(default_factory=list, description="Risques réglementaires mentionnés (ex: sanctions, compliance, nouvelles réglementations)")
    
    geopolitical_exposure: dict = Field(description="Exposition géopolitique et pays à risque", default_factory=dict)
    # Structure: {
    #   "high_risk_countries": ["China", "Russia", ...],  # Pays avec risques géopolitiques mentionnés
    #   "trade_war_exposure": True/False,  # Exposition aux tensions commerciales
    #   "sanction_risks": ["Mention of Iran sanctions", "China trade restrictions", ...]
    # }

# Modèle pour données quantitatives (Item 7, 8)
class QuantitativeData(BaseModel):
    """Données quantitatives extraites depuis un 10-K (Item 7, 8)"""
    ticker: str
    revenue_by_region: dict = Field(default_factory=dict, description="Revenue par région (pourcentages)")
    assets_by_region: dict = Field(default_factory=dict, description="Assets par région (pourcentages)")
    revenue_by_segment: dict = Field(default_factory=dict, description="Revenue par segment (pourcentages)")
    supplier_concentration: dict = Field(default_factory=dict, description="Concentration par fournisseur (pourcentages)")
    key_suppliers: List[str] = Field(default_factory=list, description="Noms des fournisseurs clés mentionnés")

print("✅ Modèles de données définis (CompanyDataPoints + QuantitativeData)")


✅ Modèles de données définis (CompanyDataPoints + QuantitativeData)


## Étape 3 : Configuration Bedrock


In [3]:
# Configuration Credentials AWS
import os
from pathlib import Path
from dotenv import load_dotenv

# Charger les variables d'environnement depuis le fichier .env
# Le fichier .env est ignoré par Git pour des raisons de sécurité
env_path = Path.cwd()

# Si on est dans un sous-répertoire (notebooks), remonter à la racine
if 'notebooks' in str(env_path):
    env_path = env_path.parent

# Chercher le fichier .env à la racine du projet
env_file = env_path / '.env'
if env_file.exists():
    load_dotenv(env_file)
    print("✅ Credentials AWS chargés depuis .env")
else:
    # Fallback: utiliser les variables d'environnement système si définies
    print("⚠️ Fichier .env non trouvé. Utilisation des variables d'environnement système.")
    if not os.environ.get('AWS_ACCESS_KEY_ID'):
        raise FileNotFoundError("Veuillez créer un fichier .env à la racine du projet ou définir les variables d'environnement AWS.")

# Initialiser Bedrock client
aws_region = os.environ.get('AWS_DEFAULT_REGION', 'us-west-2')

bedrock_client = boto3.client('bedrock-runtime', region_name=aws_region)
client = instructor.from_bedrock(bedrock_client)

# Modèles à utiliser
MODEL_SONNET = "global.anthropic.claude-sonnet-4-5-20250929-v1:0"  # Modèle principal (support 1M tokens = 4M chars)

print(f"✅ Bedrock configuré (région: {aws_region})")
print(f"   🔵 Sonnet (principal): {MODEL_SONNET} - Support jusqu'à 1M tokens (~4M chars)")
print(f"   ⚡ Si Sonnet échoue (trop long): Division automatique en 2 parties")

⚠️ Fichier .env non trouvé. Utilisation des variables d'environnement système.
✅ Bedrock configuré (région: us-west-2)
   🔵 Sonnet (principal): global.anthropic.claude-sonnet-4-5-20250929-v1:0 - Support jusqu'à 1M tokens (~4M chars)
   ⚡ Si Sonnet échoue (trop long): Division automatique en 2 parties


## Étape 4 : Fonctions d'Extraction

Cette cellule contient les deux fonctions d'extraction :
1. **`extract_quantitative_data()`** : Extrait données quantitatives (Item 7, 8)
2. **`extract_10k_data_points()`** : Fonction principale (Item 1, 1A + appelle la quantitative)

In [4]:
# Fonction 1 : Extraction quantitative (Item 7, 8)
def extract_quantitative_data(ticker: str, html_path: str) -> dict:
    """
    Extrait UNIQUEMENT les données quantitatives (pourcentages, chiffres)
    depuis Item 7 (MD&A) et Item 8 (Financial Statements)
    """
    try:
        # Lire le fichier HTML
        with open(html_path, 'r', encoding='utf-8') as f:
            html_content = f.read()
        
        # Parser avec BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')
        for script in soup(["script", "style", "meta", "link"]):
            script.decompose()
        
        # Extraire le texte complet
        text_content = soup.get_text()
        lines = [line.strip() for line in text_content.splitlines() if line.strip()]
        text_content = "\n".join(lines)
        
        # Sonnet supporte jusqu'à ~4M chars, tronquer seulement si dépasse la limite
        MAX_CHARS_SONNET = 3500000
        original_length = len(text_content)
        if original_length > MAX_CHARS_SONNET:
            text_content = text_content[:MAX_CHARS_SONNET]
            print(f"   ⚠️ Document très long ({original_length:,} → 3.5M chars)")
        
        prompt = f"""
        Tu es un expert en analyse de rapports financiers 10-K. Extrait UNIQUEMENT les données QUANTITATIVES depuis ce rapport 10-K pour {ticker}.
        
        TEXTE DU 10-K COMPLET:
        {text_content}
        
        STRATÉGIE DE RECHERCHE:
        - Cherche dans Item 7 (MD&A): revenue par région, revenue par segment
        - Cherche dans Item 8 (Financial Statements): tableaux géographiques, assets par région
        - Cherche les TABLEAUX structurés avec pourcentages/chiffres
        
        INSTRUCTIONS:
        1. **revenue_by_region**: Si tu trouves revenue/pourcentages par région
           - Format: {{"Americas": 0.427, "Europe": 0.259}}
           - Les pourcentages doivent TOTALISER ~100%
        
        2. **assets_by_region**: Si tu trouves assets par région avec pourcentages
           - Format: {{"United States": 0.55, "China": 0.30}}
           - Les pourcentages doivent TOTALISER ~100%
        
        3. **revenue_by_segment**: Si tu trouves revenue par segment avec pourcentages
           - Format: {{"iPhone": 0.52, "Services": 0.22}}
           - Les pourcentages doivent TOTALISER ~100%
        
        4. **supplier_concentration**: Si tu trouves concentration par fournisseur
           - Format: {{"Foxconn": 0.35, "TSMC": 0.25}}
        
        5. **key_suppliers**: Liste des fournisseurs clés nommés explicitement
        
        RÈGLES:
        - Extrait UNIQUEMENT les chiffres/pourcentages EXPLICITEMENT VISIBLES
        - Ne pas inventer
        - Si non trouvé, laisse VIDE {{}} ou []
        - Pourcentages en décimale (0.42 = 42%)
        
        Retourne un JSON valide selon le modèle QuantitativeData.
        """
        
        # Appel Bedrock - Essayer Sonnet d'abord, diviser si échec
        doc_length = len(text_content)
        
        # Essayer d'abord avec Sonnet
        try:
            result = client.chat.completions.create(
                modelId=MODEL_SONNET,
                messages=[{"role": "user", "content": prompt}],
                response_model=QuantitativeData,
                inferenceConfig={"maxTokens": 8000, "temperature": 0.1}
            )
            return result.model_dump()
        except Exception as e:
            # ⚡ Si Sonnet échoue (trop long) → Division récursive illimitée
            error_msg = str(e).lower()
            if "too long" in error_msg or "validationexception" in error_msg:
                print(f"   ⚠️ Sonnet échoue → Division récursive")
                
                # Fonction récursive pour division quantitative
                def extract_quant_recursive(text: str, prompt_template: str, depth: int = 0, max_depth: int = 10) -> dict:
                    """Divise récursivement jusqu'à ce que ça passe"""
                    if depth > max_depth:
                        return None
                    
                    text_len = len(text)
                    current_prompt = prompt_template.replace(text_content, text)
                    
                    try:
                        result = client.chat.completions.create(
                            modelId=MODEL_SONNET,
                            messages=[{"role": "user", "content": current_prompt}],
                            response_model=QuantitativeData,
                            inferenceConfig={"maxTokens": 8000, "temperature": 0.1}
                        )
                        return result.model_dump()
                    except Exception as e_try:
                        error_try = str(e_try).lower()
                        if "too long" in error_try or "validationexception" in error_try:
                            # Diviser
                            mid_point = len(text) // 2
                            split_point = text.rfind('\n', mid_point - 10000, mid_point + 10000)
                            if split_point == -1:
                                split_point = mid_point
                            
                            text_part1 = text[:split_point]
                            text_part2 = text[split_point:]
                            
                            data_part1 = extract_quant_recursive(text_part1, prompt_template, depth + 1, max_depth)
                            data_part2 = extract_quant_recursive(text_part2, prompt_template, depth + 1, max_depth)
                            
                            # Fusionner les données quantitatives
                            if data_part1 and data_part2:
                                merged = {}
                                for key in ['revenue_by_region', 'assets_by_region', 'revenue_by_segment', 'supplier_concentration']:
                                    d1 = data_part1.get(key, {}) or {}
                                    d2 = data_part2.get(key, {}) or {}
                                    merged[key] = {**d1, **d2}
                                merged['key_suppliers'] = list(set((data_part1.get('key_suppliers') or []) + (data_part2.get('key_suppliers') or [])))
                                return merged
                            elif data_part1:
                                return data_part1
                            elif data_part2:
                                return data_part2
                            else:
                                return None
                        elif "throttling" in error_try:
                            import time
                            wait = min(30, (depth + 1) * 5)
                            time.sleep(wait)
                            return extract_quant_recursive(text, prompt_template, depth, max_depth)
                        else:
                            return None
                
                # Lancer extraction récursive
                return extract_quant_recursive(text_content, prompt)
            else:
                print(f"   ❌ Erreur extraction quantitative: {e}")
                return None
        
    except Exception as e:
        print(f"   ⚠️ Erreur extraction quantitative pour {ticker}: {e}")
        return None

print("✅ Fonction quantitative définie")

# Fonction 2 : Extraction qualitative (Item 1, 1A) + fusion avec quantitative
def extract_10k_data_points(ticker: str, html_path: str, include_quantitative: bool = True) -> dict:
    """
    Extrait les data points depuis un fichier 10-K HTML
    STRATÉGIE OPTIMISÉE : Deux appels séparés par section
    - Appel 1 : Item 1, 1A (Business & Risks) → Données qualitatives
    - Appel 2 : Item 7, 8 (MD&A & Financial) → Données quantitatives
    
    Args:
        ticker: Ticker symbol (ex: 'AAPL')
        html_path: Chemin vers le fichier HTML 10-K
        include_quantitative: Si True, fait aussi le 2ème appel pour données quantitatives
    
    Returns:
        dict: Données structurées complètes
    """
    print(f"📄 Extraction pour {ticker}...")
    
    # 1. Lire le fichier HTML
    with open(html_path, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    # 2. Parser le HTML avec BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Extraire le texte (enlever scripts, styles, etc.)
    for script in soup(["script", "style", "meta", "link"]):
        script.decompose()
    
    # Obtenir le texte propre
    text_content = soup.get_text()
    
    # Nettoyer les espaces multiples
    lines = [line.strip() for line in text_content.splitlines() if line.strip()]
    text_content = "\n".join(lines)
    
    # Sonnet supporte jusqu'à ~4M chars, donc pas besoin de tronquer pour la plupart des documents
    original_length = len(text_content)
    MAX_CHARS_SONNET = 3500000  # Limite Sonnet ~3.5M chars
    
    if original_length > MAX_CHARS_SONNET:
        # Tronquer seulement si dépasse la limite Sonnet
        text_content = text_content[:MAX_CHARS_SONNET]
        print(f"⚠️ Document très long ({original_length:,} chars) → tronqué à 3.5M chars")
    else:
        print(f"📝 Document complet: {original_length:,} caractères")
    
    # 3. Prompt pour extraction QUALITATIVE (STRATÉGIE : Item 1, 1A - Business & Risks)
    prompt = f"""
    Tu es un expert en analyse de rapports financiers 10-K. Extrait les informations depuis ce rapport 10-K pour {ticker}.
    
    Ce texte contient le rapport 10-K complet. Cherche SPÉCIFIQUEMENT dans Item 1 (Business) et Item 1A (Risk Factors).
    
    TEXTE DU 10-K COMPLET:
    {text_content}
    
    ⚠️ IMPORTANT : Dans ce texte, trouve et utilise UNIQUEMENT les sections :
    - Item 1 : "Business" (description de l'entreprise, segments, géographie opérationnelle)
    - Item 1A : "Risk Factors" (tous les risques)
    
    IGNORE les autres sections (Item 2, 3, 7, 8, etc.) pour ce premier appel.
    
    INSTRUCTIONS D'EXTRACTION (Informations Stratégiques & Risques):
    
    1. **company_name**: Nom complet officiel de l'entreprise (exactement comme dans le document)
    2. **company_type**: Secteur d'activité principal (ex: "Technology", "Consumer Electronics", "Software Services")
    
    3. **geography** (structure JSON - QUALITATIF SEULEMENT):
       - **regions**: Liste des régions géographiques (ex: ["North America", "Europe", "Asia Pacific", "Greater China"])
       - **countries**: Liste de TOUS les pays mentionnés où l'entreprise opère (ventes, manufacturing, bureaux)
       - **has_na**: true si présence en Amérique du Nord (USA, Canada, Mexico), false sinon
       - **has_eu**: true si présence dans l'Union Européenne (membres de l'UE), false sinon
       - **revenue_by_region**: LAISSE VIDE {{}} - sera rempli par un deuxième appel spécialisé
       - **assets_by_region**: LAISSE VIDE {{}} - sera rempli par un deuxième appel spécialisé
       
    4. **segments**: Liste de TOUS les segments d'affaires mentionnés (ex: ["iPhone", "Mac", "iPad", "Services", "Wearables"] pour Apple)
    
    5. **supply_chain** (structure JSON - QUALITATIF SEULEMENT):
       - **key_suppliers**: Noms des fournisseurs clés SI MENTIONNÉS EXPLICITEMENT (ex: ["Foxconn", "TSMC"])
       - **manufacturing_locations**: Pays/régions où la manufacturing a lieu (ex: ["China", "USA", "India"])
       - **dependencies**: Dépendances critiques mentionnées (ex: "Single-source supplier for chips in Taiwan")
       - **risks**: Risques supply chain identifiés (ex: "Concentration risk in China", "Geopolitical tensions")
       - **supplier_concentration**: LAISSE VIDE {{}} - sera rempli par un deuxième appel spécialisé
    
    6. **regulatory_risks**: Liste de TOUS les risques réglementaires mentionnés (ex: "New EU regulations on data privacy", "Sanctions compliance", "FDA approval requirements", "Trade restrictions")
    
    7. **geopolitical_exposure** (structure JSON):
       - **high_risk_countries**: Pays avec risques géopolitiques explicitement mentionnés (ex: ["China", "Russia", "Iran"])
       - **trade_war_exposure**: true si tensions commerciales mentionnées (ex: "US-China trade tensions"), false sinon
       - **sanction_risks**: Liste des risques de sanctions mentionnés (ex: "Iran sanctions impact", "Russia sanctions exposure")
    
    STRATÉGIE DE RECHERCHE:
    - Cherche dans Item 1 (Business): segments, géographie opérationnelle, supply chain description
    - Cherche dans Item 1A (Risk Factors): risques réglementaires, géopolitiques, supply chain risks
    - NE PAS chercher les pourcentages/chiffres (c'est pour le 2ème appel sur Item 7, 8)
    
    RÈGLES IMPORTANTES:
    - Focus sur les informations QUALITATIVES (texte, listes, booléens)
    - NE PAS extraire de pourcentages ou chiffres financiers (Item 7, 8 gèrent ça)
    - Extrait UNIQUEMENT les informations explicitement mentionnées dans le texte
    - Si une information n'est pas trouvée, laisse le champ vide ou null (ne pas inventer)
    - Pour geography.countries: liste TOUS les pays mentionnés dans Item 1 (Business), même mineurs
    - Pour supply_chain.key_suppliers: inclure seulement les fournisseurs NOMMÉS explicitement dans Item 1
    - Pour regulatory_risks: chercher principalement dans Item 1A (Risk Factors)
    - Pour geopolitical_exposure: chercher les mentions de tensions, sanctions, risques géopolitiques dans Item 1A
    
    Retourne UNIQUEMENT un JSON valide qui correspond exactement au modèle Pydantic CompanyDataPoints.
    """
    
    # 4. Extraction avec Bedrock - Essayer Sonnet d'abord, diviser si échec
    doc_length = len(text_content)
    print(f"📝 Document: {doc_length:,} caractères → Essai avec Sonnet")
    
    # Fonction helper pour fusionner deux résultats
    def ensure_list(value):
        """Garantit qu'une valeur est toujours une liste (normalise string/None/list)"""
        if value is None:
            return []
        if isinstance(value, str):
            return [value] if value else []
        if isinstance(value, list):
            return value
        return [value]  # Pour tout autre type, le mettre dans une liste
    
    def merge_extraction_results(data1, data2):
        """Fusionne deux résultats d'extraction avec normalisation des types"""
        merged = data1.copy()
        
        # Company name : prendre le plus complet
        if data2.get('company_name') and (not merged.get('company_name') or len(data2['company_name']) > len(merged['company_name'])):
            merged['company_name'] = data2['company_name']
        
        # Lists : union (éviter doublons) - avec normalisation
        if data2.get('segments'):
            list1 = ensure_list(merged.get('segments'))
            list2 = ensure_list(data2.get('segments'))
            merged['segments'] = list(set(list1 + list2))
        if data2.get('regulatory_risks'):
            list1 = ensure_list(merged.get('regulatory_risks'))
            list2 = ensure_list(data2.get('regulatory_risks'))
            merged['regulatory_risks'] = list(set(list1 + list2))
        
        # Geography : merger avec normalisation
        if data2.get('geography'):
            geo2 = data2['geography']
            geo1 = merged.get('geography', {})
            merged['geography'] = {
                'regions': list(set(ensure_list(geo1.get('regions')) + ensure_list(geo2.get('regions')))),
                'countries': list(set(ensure_list(geo1.get('countries')) + ensure_list(geo2.get('countries')))),
                'has_na': geo1.get('has_na') or geo2.get('has_na'),
                'has_eu': geo1.get('has_eu') or geo2.get('has_eu'),
                'revenue_by_region': {**(geo1.get('revenue_by_region') or {}), **(geo2.get('revenue_by_region') or {})},
                'assets_by_region': {**(geo1.get('assets_by_region') or {}), **(geo2.get('assets_by_region') or {})}
            }
        
        # Supply chain : merger avec normalisation
        if data2.get('supply_chain'):
            sc2 = data2['supply_chain']
            sc1 = merged.get('supply_chain', {})
            merged['supply_chain'] = {
                'key_suppliers': list(set(ensure_list(sc1.get('key_suppliers')) + ensure_list(sc2.get('key_suppliers')))),
                'manufacturing_locations': list(set(ensure_list(sc1.get('manufacturing_locations')) + ensure_list(sc2.get('manufacturing_locations')))),
                'dependencies': sc1.get('dependencies') or sc2.get('dependencies'),
                'risks': list(set(ensure_list(sc1.get('risks')) + ensure_list(sc2.get('risks')))),
                'supplier_concentration': {**(sc1.get('supplier_concentration') or {}), **(sc2.get('supplier_concentration') or {})}
            }
        
        # Geopolitical exposure : merger avec normalisation
        if data2.get('geopolitical_exposure'):
            gp2 = data2['geopolitical_exposure']
            gp1 = merged.get('geopolitical_exposure', {})
            merged['geopolitical_exposure'] = {
                'high_risk_countries': list(set(ensure_list(gp1.get('high_risk_countries')) + ensure_list(gp2.get('high_risk_countries')))),
                'trade_war_exposure': gp1.get('trade_war_exposure') or gp2.get('trade_war_exposure'),
                'sanction_risks': list(set(ensure_list(gp1.get('sanction_risks')) + ensure_list(gp2.get('sanction_risks'))))
            }
        
        return merged
    
    # Essayer d'abord avec Sonnet (même si document long)
    try:
        result = client.chat.completions.create(
            modelId=MODEL_SONNET,
            messages=[{"role": "user", "content": prompt}],
            response_model=CompanyDataPoints,
            inferenceConfig={"maxTokens": 8000, "temperature": 0.1}
        )
        
        data = result.model_dump()
        data['ticker'] = ticker
        print(f"✅ Extraction réussie avec Sonnet pour {ticker}")
        
    except Exception as e:
        # ⚡ Si Sonnet échoue (trop long) → Division récursive avec limite
        error_msg = str(e).lower()
        if "too long" in error_msg or "validationexception" in error_msg:
            print(f"⚠️ Sonnet échoue (document trop long: {doc_length:,} chars) → Division récursive")
            
            # Fonction récursive pour diviser et extraire (max 10 divisions)
            def extract_recursive(text: str, prompt_template: str, depth: int = 0, max_depth: int = 10) -> dict:
                """Divise récursivement jusqu'à ce que ça passe (max_depth limité)"""
                if depth > max_depth:
                    print(f"   {'  ' * depth}❌ Profondeur max ({max_depth}) atteinte, arrêt")
                    return None
                
                text_len = len(text)
                current_prompt = prompt_template.replace(text_content, text)
                
                # Essayer d'extraire
                try:
                    result = client.chat.completions.create(
                        modelId=MODEL_SONNET,
                        messages=[{"role": "user", "content": current_prompt}],
                        response_model=CompanyDataPoints,
                        inferenceConfig={"maxTokens": 8000, "temperature": 0.1}
                    )
                    print(f"   {'  ' * depth}✅ Partie extraite ({text_len:,} chars)")
                    return result.model_dump()
                except Exception as e_try:
                    error_try = str(e_try).lower()
                    if "too long" in error_try or "validationexception" in error_try:
                        # Encore trop long → diviser en 2
                        print(f"   {'  ' * depth}⚠️ Encore trop long ({text_len:,} chars) → Division")
                        
                        # Diviser
                        mid_point = text_len // 2
                        split_point = text.rfind('\n', mid_point - 10000, mid_point + 10000)
                        if split_point == -1:
                            split_point = mid_point
                        
                        text_part1 = text[:split_point]
                        text_part2 = text[split_point:]
                        
                        # Extraire récursivement chaque partie
                        data_part1 = extract_recursive(text_part1, prompt_template, depth + 1, max_depth)
                        data_part2 = extract_recursive(text_part2, prompt_template, depth + 1, max_depth)
                        
                        # Fusionner
                        if data_part1 and data_part2:
                            merged = merge_extraction_results(data_part1, data_part2)
                            return merged
                        elif data_part1:
                            return data_part1
                        elif data_part2:
                            return data_part2
                        else:
                            return None
                    elif "throttling" in error_try:
                        # Gérer throttling avec délai exponentiel
                        import time
                        wait = min(30, (depth + 1) * 5)
                        print(f"   {'  ' * depth}⏳ Throttling, attente {wait}s...")
                        time.sleep(wait)
                        return extract_recursive(text, prompt_template, depth, max_depth)  # Retry
                    else:
                        print(f"   {'  ' * depth}⚠️ Erreur: {e_try}")
                        return None
            
            # Lancer l'extraction récursive
            extracted_data = extract_recursive(text_content, prompt)
            
            if extracted_data:
                extracted_data['ticker'] = ticker
                data = extracted_data
                print(f"✅ Extraction réussie avec division récursive pour {ticker}")
            else:
                print(f"❌ Erreur extraction {ticker}: échec après division récursive (profondeur max atteinte)")
                return None
        else:
            print(f"❌ Erreur extraction {ticker}: {e}")
            return None
    
    # ⚠️ IMPORTANT : Faire le 2ème appel AVANT de retourner
    # Si include_quantitative, faire le 2ème appel pour données quantitatives
    if include_quantitative and data:
        quantitative_data = extract_quantitative_data(ticker, html_path)
        if quantitative_data:
            # Fusionner les données quantitatives directement
            data['geography']['revenue_by_region'] = quantitative_data.get('revenue_by_region', {})
            data['geography']['assets_by_region'] = quantitative_data.get('assets_by_region', {})
            data['supply_chain']['supplier_concentration'] = quantitative_data.get('supplier_concentration', {})
            
            # Key suppliers: merger (éviter doublons)
            main_suppliers = set(data['supply_chain'].get('key_suppliers', []))
            quant_suppliers = set(quantitative_data.get('key_suppliers', []))
            data['supply_chain']['key_suppliers'] = list(main_suppliers | quant_suppliers)
            
            # Ajouter revenue_by_segment si disponible
            revenue_seg = quantitative_data.get('revenue_by_segment', {})
            if revenue_seg:
                data['revenue_by_segment'] = revenue_seg
            
            print(f"✅ Données fusionnées pour {ticker}")
    
    return data

print("✅ Toutes les fonctions d'extraction définies (quantitative + qualitative)")


✅ Fonction quantitative définie
✅ Toutes les fonctions d'extraction définies (quantitative + qualitative)


## Étape 5 : Test sur un Fichier (Exemple AAPL)


In [6]:
# Test sur Apple (AAPL)
test_ticker = "AAPL"
test_path = f"../../data/raw/fillings/{test_ticker}/2024-11-01-10k-{test_ticker}.html"

if os.path.exists(test_path):
    print("📋 Stratégie: Deux appels (Item 1,1A + Item 7,8)")
    result = extract_10k_data_points(test_ticker, test_path, include_quantitative=True)
    if result:
        print("\n📊 Résultat extrait:")
        print(json.dumps(result, indent=2, ensure_ascii=False)[:2000] + "...")
        
        # Sauvegarder en JSON
        os.makedirs("../../data/generated/extracted_data_points", exist_ok=True)
        output_file = f"../../data/generated/extracted_data_points/{test_ticker}_data_points.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        print(f"\n💾 Données sauvegardées: {output_file}")
else:
    print(f"❌ Fichier non trouvé: {test_path}")
    print("💡 Cherchez le bon fichier dans ../../data/raw/fillings/")


📋 Stratégie: Deux appels (Item 1,1A + Item 7,8)
📄 Extraction pour AAPL...
📝 Document complet: 215,214 caractères
📝 Document: 215,214 caractères → Essai avec Sonnet
✅ Extraction réussie avec Sonnet pour AAPL
✅ Données fusionnées pour AAPL

📊 Résultat extrait:
{
  "ticker": "AAPL",
  "company_name": "Apple Inc.",
  "company_type": "Technology - Consumer Electronics and Software Services",
  "geography": {
    "regions": [
      "Americas",
      "Europe",
      "Greater China",
      "Japan",
      "Rest of Asia Pacific"
    ],
    "countries": [
      "United States",
      "Canada",
      "Mexico",
      "China",
      "Hong Kong",
      "Taiwan",
      "Japan",
      "South Korea",
      "Vietnam",
      "India",
      "Australia",
      "Singapore",
      "Ireland",
      "Middle East countries",
      "African countries",
      "European Union member states"
    ],
    "has_na": true,
    "has_eu": true,
    "revenue_by_region": {
      "Americas": 0.427,
      "Europe": 0.259,
    

## Étape 6 : Traitement en Lot (Tous les Tickers)


In [5]:
def extract_all_data_points(
    output_dir: str = "../../data/generated/extracted_data_points",
    max_tickers: Optional[int] = None,
    start_index: Optional[int] = None,
    end_index: Optional[int] = None,
    parallel: bool = True,
    max_workers: int = 2,  # Réduit pour éviter throttling
    resume_from_cache: bool = True
):
    """
    Extrait les data points pour tous les tickers avec optimisations
    
    Args:
        output_dir: Dossier où sauvegarder les résultats
        max_tickers: Limite le nombre (None = tous). Ex: 20 pour tester
        start_index: Index de début dans la liste (ex: 0, 50, 100)
        end_index: Index de fin dans la liste (ex: 50, 100, 150). None = jusqu'à la fin
        parallel: Si True, traitement parallèle (5x plus rapide)
        max_workers: Nombre de workers (5-10 recommandé)
        resume_from_cache: Si True, skip les tickers déjà extraits
    """
    # Créer dossier de sortie
    os.makedirs(output_dir, exist_ok=True)
    
    # Liste pour stocker tous les résultats
    all_results = {}
    
    # Parcourir tous les dossiers dans fillings/
    fillings_dir = Path("../../data/raw/fillings")
    if not fillings_dir.exists():
        print("❌ Dossier ../../data/raw/fillings/ non trouvé")
        return
    
    # ⚙️ RÉCUPÉRATION TRIÉE: Garantit toujours le même ordre (alphabétique)
    tickers = sorted([d.name for d in fillings_dir.iterdir() if d.is_dir()])
    total_tickers = len(tickers)
    print(f"📋 Total tickers disponibles: {total_tickers} (triés alphabétiquement)")
    if tickers:
        print(f"   Premiers: {', '.join(tickers[:5])}...")
        print(f"   Derniers: ...{', '.join(tickers[-5:])}")
    
    # ⚙️ OPTIMISATION 1: Skip ceux déjà traités
    if resume_from_cache:
        existing_files = set(f.stem.replace('_data_points', '') 
                           for f in Path(output_dir).glob('*_data_points.json'))
        tickers = [t for t in tickers if t not in existing_files]
        if existing_files:
            print(f"⏭️  {len(existing_files)} tickers déjà traités, skip...")
    
    # ⚙️ OPTIMISATION 2: Plage spécifique (start_index, end_index)
    if start_index is not None or end_index is not None:
        start = start_index if start_index is not None else 0
        end = end_index if end_index is not None else len(tickers)
        tickers = tickers[start:end]
        print(f"📌 Plage sélectionnée: tickers [{start}:{end}] sur {total_tickers} totaux")
        print(f"   Tickers restants à traiter: {len(tickers)}")
    # Sinon, utiliser max_tickers si spécifié
    elif max_tickers:
        tickers = tickers[:max_tickers]
        print(f"📌 Limité à {max_tickers} tickers pour ce test")
    
    print(f"📋 {len(tickers)} tickers à traiter")
    
    if not tickers:
        print("✅ Tous les tickers sont déjà traités!")
        return
    
    # ⚙️ OPTIMISATION 3: Traitement parallèle
    def process_ticker(ticker: str) -> tuple:
        """Process un ticker et retourne (ticker, result)"""
        try:
            ticker_dir = fillings_dir / ticker
            html_files = list(ticker_dir.glob("*.html"))
            
            if not html_files:
                print(f"⚠️ [{ticker}] Aucun HTML")
                return (ticker, None)
            
            html_file = sorted(html_files, key=lambda x: x.stat().st_mtime, reverse=True)[0]
            result = extract_10k_data_points(ticker, str(html_file), include_quantitative=True)
            
            # ⚙️ SAUVEGARDE IMMÉDIATE : Écrit le JSON dès qu'un ticker est traité
            if result:
                # S'assurer que le dossier existe (important en traitement parallèle)
                os.makedirs(output_dir, exist_ok=True)
                output_file = os.path.join(output_dir, f"{ticker}_data_points.json")
                try:
                    # Écriture atomique : write + flush pour garantir la sauvegarde immédiate
                    with open(output_file, 'w', encoding='utf-8') as f:
                        json.dump(result, f, indent=2, ensure_ascii=False)
                        f.flush()  # Force l'écriture sur disque immédiatement
                        os.fsync(f.fileno())  # Garantit que le fichier est écrit sur disque
                    print(f"✅ [{ticker}] Sauvegardé: {output_file}")
                except Exception as e:
                    print(f"❌ [{ticker}] Erreur sauvegarde: {e}")
            
            return (ticker, result)
        except Exception as e:
            print(f"❌ [{ticker}] Erreur: {e}")
            return (ticker, None)
    
    if parallel:
        from concurrent.futures import ThreadPoolExecutor, as_completed
        
        print(f"🚀 PARALLÈLE par BATCH ({max_workers} workers par batch)...")
        
        # Diviser les tickers en batchs de max_workers
        total_batches = (len(tickers) + max_workers - 1) // max_workers
        batch_num = 0
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            for i in range(0, len(tickers), max_workers):
                batch_num += 1
                batch_tickers = tickers[i:i + max_workers]
                
                # 🔵 AFFICHAGE TRÈS VISIBLE DU DÉBUT DU BATCH
                import sys
                batch_msg = f"🚀 NOUVEAU BATCH {batch_num}/{total_batches}"
                tickers_msg = f"📦 Tickers: {', '.join(batch_tickers)}"
                remaining_msg = f"📊 Restants: {len(tickers) - (i + len(batch_tickers))}"
                
                print(f"\n\n{'#'*80}")
                print(f"{'#'*80}")
                print(f"# {batch_msg:<76} #")
                print(f"# {tickers_msg:<76} #")
                print(f"# {remaining_msg:<76} #")
                print(f"{'#'*80}")
                print(f"{'#'*80}\n\n")
                sys.stdout.flush()  # Force l'affichage immédiat
                
                # Soumettre uniquement ce batch
                future_to_ticker = {executor.submit(process_ticker, ticker): ticker 
                                    for ticker in batch_tickers}
                
                # Attendre que TOUS les tickers du batch finissent avant de continuer
                for future in as_completed(future_to_ticker):
                    ticker, result = future.result()
                    if result:
                        all_results[ticker] = result
                
                # 🔵 AFFICHAGE DE LA FIN DU BATCH
                print(f"\n{'='*60}")
                print(f"✅ BATCH {batch_num}/{total_batches} TERMINÉ ({len(batch_tickers)} tickers)")
                print(f"{'='*60}\n")
    else:
        # Traitement séquentiel (original)
        print("🐢 SÉQUENTIEL...")
        for i, ticker in enumerate(tickers, 1):
            print(f"\n[{i}/{len(tickers)}] {ticker}...")
            ticker, result = process_ticker(ticker)
            if result:
                all_results[ticker] = result
    
    # ⚙️ On ne génère PAS les fichiers consolidés ici
    # Ils seront générés à la fin avec generate_consolidated_files()
    print(f"\n✅ Extraction terminée pour {len(all_results)} tickers")
    print(f"💾 Fichiers individuels sauvegardés dans: {output_dir}/")
    
    return all_results

def generate_consolidated_files(output_dir: str = "extracted_data_points"):
    """
    Génère les fichiers consolidés (all_data_points.json + data_points.csv)
    à partir de TOUS les JSON individuels déjà créés.
    
    À appeler à la fin, quand tous les tickers sont traités.
    
    Args:
        output_dir: Dossier contenant les JSON individuels
    """
    print("\n📦 Génération des fichiers consolidés...")
    
    # Lire tous les JSON individuels
    json_files = list(Path(output_dir).glob("*_data_points.json"))
    
    # Exclure all_data_points.json s'il existe
    json_files = [f for f in json_files if f.name != "all_data_points.json"]
    
    if not json_files:
        print("⚠️ Aucun fichier JSON individuel trouvé")
        return
    
    all_results = {}
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                ticker = data.get('ticker', json_file.stem.replace('_data_points', ''))
                all_results[ticker] = data
        except Exception as e:
            print(f"⚠️ Erreur lecture {json_file.name}: {e}")
    
    print(f"📋 {len(all_results)} tickers chargés")
    
    # Sauvegarder all_data_points.json
    all_output_file = os.path.join(output_dir, "all_data_points.json")
    with open(all_output_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    print(f"✅ Fichier consolidé JSON: {all_output_file}")
    
    # Convertir en CSV
    convert_to_csv(all_results, output_dir)
    
    return all_results

def convert_to_csv(results: dict, output_dir: str):
    """Convertit les résultats JSON en CSV (enrichi)"""
    rows = []
    for ticker, data in results.items():
        geo = data.get('geography', {})
        sc = data.get('supply_chain', {})
        geo_exp = data.get('geopolitical_exposure', {})
        
        row = {
            'ticker': ticker,
            'company_name': data.get('company_name', ''),
            'company_type': data.get('company_type', ''),
            
            # Géographie
            'geography_regions': '; '.join(geo.get('regions', [])),
            'geography_countries': '; '.join(geo.get('countries', [])),
            'has_na': geo.get('has_na', False),
            'has_eu': geo.get('has_eu', False),
            'revenue_by_region': json.dumps(geo.get('revenue_by_region', {})) if geo.get('revenue_by_region') else '',
            'assets_by_region': json.dumps(geo.get('assets_by_region', {})) if geo.get('assets_by_region') else '',
            
            # Segments
            'segments': '; '.join(data.get('segments', [])),
            
            # Supply Chain
            'supply_chain_suppliers': '; '.join(sc.get('key_suppliers', [])),
            'manufacturing_locations': '; '.join(sc.get('manufacturing_locations', [])),
            'supply_chain_dependencies': '; '.join(sc.get('dependencies', [])),
            'supply_chain_risks': '; '.join(sc.get('risks', [])),
            'supplier_concentration': json.dumps(sc.get('supplier_concentration', {})) if sc.get('supplier_concentration') else '',
            
            # Réglementaire
            'regulatory_risks': '; '.join(data.get('regulatory_risks', [])),
            
            # Géopolitique
            'high_risk_countries': '; '.join(geo_exp.get('high_risk_countries', [])),
            'trade_war_exposure': geo_exp.get('trade_war_exposure', False),
            'sanction_risks': '; '.join(geo_exp.get('sanction_risks', [])),
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    csv_file = os.path.join(output_dir, "data_points.csv")
    df.to_csv(csv_file, index=False)
    print(f"📊 CSV sauvegardé: {csv_file}")
    print(f"   Colonnes: {df.columns.tolist()}")
    print(f"   Lignes: {len(df)}")


In [8]:
# 🔍 Vérifier l'état d'avancement (combien de tickers traités)
import os
from pathlib import Path

# Récupérer TOUS les tickers dans le même ordre (triés alphabétiquement)
fillings_dir = Path("../../data/raw/fillings")
all_tickers = sorted([d.name for d in fillings_dir.iterdir() if d.is_dir()])

# Exclure all_data_points.json du comptage (fichier consolidé)
extracted_files = [f for f in Path("../../data/generated/extracted_data_points").glob("*_data_points.json") 
               if f.name != "all_data_points.json"]
tickers_done = sorted([f.stem.replace('_data_points', '') for f in extracted_files])

print(f"✅ Tickers déjà extraits: {len(extracted_files)}")
print(f"📋 Total tickers disponibles: {len(all_tickers)}")
print(f"📊 Reste à traiter: {len(all_tickers) - len(extracted_files)}")

# Vérifier si fichiers consolidés existent
consolidated_json = Path("../../data/generated/extracted_data_points/all_data_points.json")
consolidated_csv = Path("../../data/generated/extracted_data_points/data_points.csv")
has_json = consolidated_json.exists()
has_csv = consolidated_csv.exists()

if has_json or has_csv:
    print(f"\n📦 Fichiers consolidés existants:")
    if has_json:
        print(f"   ✅ all_data_points.json")
    if has_csv:
        print(f"   ✅ data_points.csv")
else:
    print(f"\n💡 Pour générer les fichiers consolidés, lance la Cellule 17 (après tous les tickers)")

# Afficher l'ordre (premiers et derniers)
if all_tickers:
    print(f"\n📝 Ordre des tickers (alphabétique):")
    print(f"   Premiers: {', '.join(all_tickers[:5])}...")
    print(f"   Derniers: ...{', '.join(all_tickers[-5:])}")

# Suggérer le prochain start_index basé sur l'ordre
if tickers_done:
    print(f"\n📝 Derniers tickers traités: {', '.join(tickers_done[-5:])}")
    
    # Trouver le dernier index traité
    last_ticker = tickers_done[-1]
    if last_ticker in all_tickers:
        last_index = all_tickers.index(last_ticker)
        next_start = last_index + 1
        next_end = min(next_start + 50, len(all_tickers))
        print(f"\n💡 Prochain batch recommandé:")
        print(f"   start_index={next_start}, end_index={next_end}")
        print(f"   (Traitera les tickers de {next_start} à {next_end-1})")
    else:
        print(f"\n⚠️ Dernier ticker '{last_ticker}' non trouvé dans la liste triée")
else:
    print(f"\n💡 Commencez avec: start_index=0, end_index=50")


✅ Tickers déjà extraits: 318
📋 Total tickers disponibles: 500
📊 Reste à traiter: 182

📦 Fichiers consolidés existants:
   ✅ all_data_points.json

📝 Ordre des tickers (alphabétique):
   Premiers: A, AAPL, ABBV, ABNB, ABT...
   Derniers: ...XYZ, YUM, ZBH, ZBRA, ZTS

📝 Derniers tickers traités: NVDA, NVR, NWS, NWSA, NXPI

💡 Prochain batch recommandé:
   start_index=345, end_index=395
   (Traitera les tickers de 345 à 394)


## Étape 7 : Extraction Complète (Tous les Tickers)


In [9]:
# ⚙️ CONFIGURATION - Modifiez selon vos besoins
# 💡 TIP: Lancez d'abord la cellule ci-dessous pour voir l'état d'avancement !

all_results = extract_all_data_points(
    start_index=345,           
    end_index=500,             
    parallel=True,
    max_workers=20,
    resume_from_cache=False   
)


📋 Total tickers disponibles: 500 (triés alphabétiquement)
   Premiers: A, AAPL, ABBV, ABNB, ABT...
   Derniers: ...XYZ, YUM, ZBH, ZBRA, ZTS
📌 Plage sélectionnée: tickers [345:500] sur 500 totaux
   Tickers restants à traiter: 155
📋 155 tickers à traiter
🚀 PARALLÈLE par BATCH (20 workers par batch)...


################################################################################
################################################################################
# 🚀 NOUVEAU BATCH 1/8                                                          #
# 📦 Tickers: O, ODFL, OKE, OMC, ON, ORCL, ORLY, OTIS, OXY, PANW, PAYC, PAYX, PCAR, PCG, PEG, PEP, PFE, PFG, PG, PGR #
# 📊 Restants: 135                                                              #
################################################################################
################################################################################


📄 Extraction pour O...
📄 Extraction pour OKE...
📄 Extraction pour ODFL...
📄 Extraction pou